# Woolworths NZ Meal Cost Optimizer

Find the cheapest Woolworths for your recipe by comparing ingredient prices across nearby stores.

## Setup
Ensure you have the required dependencies and that the Woolworths store data is available in the `data/` directory.

In [7]:
import sys
import pandas as pd
import math
import requests
import time
import os
import asyncio
from pathlib import Path
from playwright.async_api import async_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

%load_ext autoreload
%autoreload 2

# Ensure project root is in path to allow relative package imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from scripts.woolworths.woolworths_optimizer import geocode, haversine, load_and_filter_stores, get_ingredients, get_quantities, change_store, scrape_products, analyze_results

import nest_asyncio
nest_asyncio.apply()

# ═══════ YOUR INPUTS ═══════
USER_ADDRESS = "123 Queen Street, Auckland CBD, 1010"
DISH_NAME = "spaghetti bolognese"
MAX_DISTANCE_KM = 5
# ═══════════════════════════

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Geocode and Filter Stores
Locate stores within the specified radius of your address.

In [8]:
user_lat, user_lon = geocode(USER_ADDRESS)

# Debug and construct robust absolute path
notebook_dir = os.getcwd()
print(f"DEBUG: CWD is {notebook_dir}")
stores_csv_path = os.path.abspath(os.path.join(notebook_dir, '..', 'data', 'woolworths_stores.csv'))
print(f"DEBUG: Constructed path is {stores_csv_path}")

stores = load_and_filter_stores(user_lat, user_lon, stores_csv_path=stores_csv_path, max_dist_km=MAX_DISTANCE_KM).head(2)
ingredients = get_ingredients(DISH_NAME)
quantities = get_quantities(DISH_NAME)

print(f"Dish: {DISH_NAME.title()}")
print(f"Ingredients: {', '.join(ingredients)}")
print(f"Stores to check: {', '.join(stores['name'])}")

DEBUG: CWD is c:\Users\bryce\Python Projects\OpenCode\notebooks
DEBUG: Constructed path is c:\Users\bryce\Python Projects\OpenCode\data\woolworths_stores.csv
Dish: Spaghetti Bolognese
Ingredients: beef mince, spaghetti pasta, canned tomatoes, onion, carrot, garlic, mixed herbs
Stores to check: Woolworths Ponsonby, Woolworths Auckland Quay Street


## 2. Run Scraper
This runs the scraper as a standalone process and streams the output in real-time.

In [9]:
import subprocess

results_file = "../data/latest_results.csv"
# Ensure file doesn't exist so we know it's being regenerated
if os.path.exists(results_file):
    os.remove(results_file)

cmd = [sys.executable, "../scripts/woolworths/woolworths_optimizer.py", USER_ADDRESS, DISH_NAME, results_file]

# Run and stream output in real-time
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()


--- Store: Woolworths Ponsonby ---
  Scraping: beef mince
    Found: Woolworths NZ Beef Mince Grass Fed 5% Fat Tray 500g - $16.99 each. ($33.98 / 1kg)
    Found: Woolworths NZ Beef Mince Grass Fed 18% Fat Tray 1kg - $18.99 each. ($18.99 / 1kg)
    Found: Woolworths Pork & Beef Mince Prepacked 300g - $9.00 each. ($30.00 / 1kg)
    Found: Woolworths NZ Beef Mince Grass Fed 18% Fat Tray 500g - $12.99 each. ($25.98 / 1kg)
    Found: Woolworths NZ Beef Mince Prime Grass Fed 13% Fat Tray 750g - $19.99 each. ($26.65 / 1kg)
    Found: First Light Beef Mince Wagyu 400g - $13.49 each. ($33.73 / 1kg)
    Found: Woolworths NZ Beef Mince Grass Fed 5% Fat Tray 1kg - $30.99 each. ($30.99 / 1kg)
    Found: Woolworths NZ Angus Beef Mince Grass Fed 10% Fat Tray 500g - $15.99 each. ($31.98 / 1kg)
    Found: Woolworths Essentials Beef Mince Value 1.8kg Pack - $31.90 each. ($17.72 / 1kg)
    Found: Woolworths NZ Beef Meatballs 20 Pack Tray 400g - $9.99 each. ($24.98 / 1kg)
    Found: Woolworths Beef Stack

0

## 3. Analysis and Comparison
Load results and display tables.

In [10]:
df = pd.read_csv(results_file)
summary, table = analyze_results(df, ingredients, DISH_NAME)

print("=" * 65)
print("COST COMPARISON (Cheapest items only)")
print("=" * 65)
print(summary.to_string())
print("\nPer-Ingredient Breakdown:")
display(table)

COST COMPARISON (Cheapest items only)
                                 total_cost
store                                      
Woolworths Auckland Quay Street       19.25
Woolworths Ponsonby                   19.25

Per-Ingredient Breakdown:


,Qty,Woolworths Auckland Quay Street,Woolworths Ponsonby,Best Price,Best Store
Ingredient,,,,,
beef mince,500g,Woolworths Pork & Be...: $9.00 ($30.00 / 1kg),Woolworths Pork & Be...: $9.00 ($30.00 / 1kg),$9.00,Woolworths Auckland Quay Street
spaghetti pasta,400g,Essentials Pasta Spa...: $1.65 ($0.33 / 100g),Essentials Pasta Spa...: $1.65 ($0.33 / 100g),$1.65,Woolworths Auckland Quay Street
canned tomatoes,1 can (400g),Essentials Diced Tom...: $0.97 ($2.43 / 1kg),Essentials Diced Tom...: $0.97 ($2.43 / 1kg),$0.97,Woolworths Auckland Quay Street
onion,1 medium,Fresh Vegetable Onio...: $1.69 ($1.69 / 1kg),Fresh Vegetable Onio...: $1.69 ($1.69 / 1kg),$1.69,Woolworths Auckland Quay Street
carrot,2 medium,Fresh Vegetable Carr...: $1.95 ($1.95 / 1kg),Fresh Vegetable Carr...: $1.95 ($1.95 / 1kg),$1.95,Woolworths Auckland Quay Street
garlic,2 cloves,Woolworths Minced Ga...: $2.70 ($0.11 / 10g),Woolworths Minced Ga...: $2.70 ($0.11 / 10g),$2.70,Woolworths Auckland Quay Street
mixed herbs,1 tsp,Maggi Instant Gravy ...: $1.29 ($0.48 / 10g),Maggi Instant Gravy ...: $1.29 ($0.48 / 10g),$1.29,Woolworths Auckland Quay Street
TOTAL,,$19.25,$19.25,$19.25,(mix)
